# ARIMA

Mỗi KPI được chọn order riêng `(p,d,q)` bằng AIC và ngưỡng riêng (tune trên val).

**Quy trình mỗi KPI:**
1. `time_split_per_kpi` 60/20/20 theo thời gian → `preprocess_all` (robust, mask gap dài thành segment). Bỏ KPI mà val hoặc test không có anomaly.
2. **Khử mùa vụ** theo chu kỳ ngày (period = 86400/step: 1440 cho 60s, 288 cho 300s) — profile = median theo time-of-day, fit **chỉ trên train**.
3. **Chọn d** per-KPI bằng ADF (trên chuỗi đã khử mùa vụ), rồi **chọn (p,q)** bằng AIC grid. Cố định d trước để AIC so sánh được giữa các ứng viên.
4. **Residual one-step-ahead** theo từng segment (không bắc cầu qua gap):
    - `q=0 & d=0` → đệ quy AR vectorized `O(n)` (đúng phương pháp notebook đơn KPI).
    - còn lại → statsmodels `.apply(refit=False)` tái dùng tham số, lấy dự báo 1 bước.
5. **Score = |resid| / σ**, với σ = MAD robust trên residual train (giống ARIMA đơn + STL → so sánh công bằng giữa các model).
6. `evaluate_protocol`: tune threshold trên **val**, đo trên **test** (PW/PA/AP/TTD).

In [1]:
import sys, time, warnings, os
from pathlib import Path

def find_root(marker='Data/train.csv'):
    for d in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (d / marker).exists():
            return d
    raise FileNotFoundError(f'Không thấy repo root (marker {marker}) từ {Path.cwd()}')

ROOT = find_root()
sys.path.append(str(ROOT / 'Modeling' / 'Code'))
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller
from eval_protocol import time_split_per_kpi, evaluate_protocol
from preprocess import preprocess_all

In [2]:
df = pd.read_csv(ROOT / 'Data' / 'train.csv')
df.columns = ['timestamp', 'value', 'label', 'kpi']

P_RANGE = range(0, 4)
Q_RANGE = range(0, 3)
N_FIT   = 6000
N_STAT  = 8000
MAX_GAP = 5                
print('Repo root:', ROOT)
print('train.csv:', df.kpi.nunique())

Repo root: C:\Projects\anomaly-detection-fundamentals
train.csv: 26


## Hàm phụ trợ

- `choose_d` — d=1 nếu ADF nói chuỗi (đã khử mùa vụ) chưa dừng, ngược lại d=0.
- `deseason` — khử nhịp ngày bằng profile median theo time-of-day (fit train).
- `resid_vectorized` / `resid_statespace` — hai cách tính residual 1 bước theo segment (xem header).

In [3]:
def choose_d(s):
    # d=1 nếu chuỗi (đã khử mùa vụ) chưa dừng theo ADF; ngược lại d=0.
    # Cố định d trước rồi mới grid (p,q) để AIC so sánh được giữa các ứng viên.
    s = s.dropna()
    if len(s) > N_STAT:
        s = s.iloc[-N_STAT:]
    try:
        adf_p = adfuller(s, autolag='AIC')[1]
    except Exception:
        adf_p = 1.0
    return 1 if adf_p >= 0.05 else 0


def deseason(p, step):
    # Khử nhịp ngày: profile = median theo time-of-day, FIT chỉ trên train.
    period = int(round(86400 / step))                       # 1440 cho 60s, 288 cho 300s
    tod = ((p.timestamp.values % 86400) // step).astype(int) % period
    xn = p.value_norm.values.astype(float)
    trm = (p.split.values == 'train') & p.value_filled.notna().values
    prof = pd.Series(xn[trm]).groupby(tod[trm]).median()
    prof = prof.reindex(range(period)).interpolate().bfill().ffill()
    seasonal = pd.Series(tod).map(prof).values
    return xn - seasonal, period


def resid_vectorized(ds, segs, mu, phi):
    # Dự báo AR 1 bước theo từng segment (O(n)): pred = mu + Σ φ_i·(y_{t-i} - mu).
    ds = pd.Series(ds)
    segs = pd.Series(segs)
    pred = pd.Series(mu, index=ds.index, dtype=float)
    valid = ds.notna()
    for i, ph in enumerate(phi, start=1):
        lag = ds.groupby(segs).shift(i)
        pred = pred + ph * (lag - mu)
        valid = valid & lag.notna()
    return (ds - pred).where(valid).values


def resid_statespace(ds, segs, res0, order):
    # Có MA hoặc sai phân: dùng .apply(refit=False) tái dùng tham số, lấy dự báo 1 bước theo segment.
    resid = np.full(len(ds), np.nan)
    minlen = max(order[0] + order[2] + order[1] + 2, 10)
    for sid in pd.unique(segs):
        idx = np.where((segs == sid) & ~np.isnan(ds))[0]
        if len(idx) < minlen:
            continue
        seg = ds[idx]
        try:
            rs = res0.apply(seg, refit=False)
            pm = rs.get_prediction(start=0, end=len(seg) - 1, dynamic=False).predicted_mean
            resid[idx] = seg - np.asarray(pm)
        except Exception:
            continue
    return resid

## Hàm chạy 1 KPI — chọn order + ngưỡng RIÊNG cho từng KPI

In [4]:
def run_kpi(gk, max_gap_points=MAX_GAP):
    step = int(pd.Series(np.diff(np.sort(gk.timestamp.values))).mode().iloc[0])
    gk = time_split_per_kpi(gk, 0.6, 0.2)
    p = (preprocess_all(gk, max_gap_points=max_gap_points, norm_method='robust')
         .sort_values('timestamp').reset_index(drop=True))
    y = p.label.values.astype(int); spl = p.split.values
    if y[spl == 'val'].sum() == 0 or y[spl == 'test'].sum() == 0:
        return None
    ds, period = deseason(p, step)
    p['deseason'] = ds
    segs = p.segment.values

    tr_real = p[(spl == 'train') & p.deseason.notna()]
    if len(tr_real) == 0:
        return None
    seg_id = tr_real.groupby('segment').size().idxmax()
    s_train = p.loc[(segs == seg_id) & (spl == 'train') & p.deseason.notna(), 'deseason'].reset_index(drop=True)
    if len(s_train) < 200:
        return None
    d = choose_d(s_train)
    s_fit = s_train.iloc[-N_FIT:] if len(s_train) > N_FIT else s_train

    best = None; best_res = None
    for pp in P_RANGE:
        for qq in Q_RANGE:
            if pp == 0 and qq == 0:
                continue
            try:
                with warnings.catch_warnings():
                    warnings.simplefilter('ignore')
                    r = ARIMA(s_fit, order=(pp, d, qq)).fit()
                if best is None or r.aic < best[0]:
                    best = (r.aic, pp, qq); best_res = r
            except Exception:
                continue
    if best_res is None:
        return None
    aic, P, Q = best

    if d == 0 and Q == 0:
        prm = best_res.params
        mu = float(prm.get('const', 0.0))
        phi = [float(prm['ar.L' + str(i)]) for i in range(1, P + 1)]
        resid = resid_vectorized(p.deseason.values, segs, mu, phi); path = 'vec'
    else:                                                 # có MA/sai phân → statsmodels
        resid = resid_statespace(p.deseason.values, segs, best_res, (P, d, Q)); path = 'sm'

    p['resid'] = resid                                    # score = |resid|/σ, σ = MAD robust trên train
    rtr = p.loc[(spl == 'train') & p.resid.notna(), 'resid']
    med = rtr.median(); sigma = np.median(np.abs(rtr - med)) * 1.4826
    sigma = sigma if sigma > 1e-9 else 1e-9
    score = np.abs(resid) / sigma
    valid = ~np.isnan(score)
    vm = (spl == 'val') & valid; tm = (spl == 'test') & valid
    if y[vm].sum() == 0 or y[tm].sum() == 0:
        return None
    te = (spl == 'test')
    r = evaluate_protocol(y[vm], score[vm], y[tm], score[tm], step_s=step,
                          y_test_full=y[te], valid_test=valid[te])   # fix TTD + P/R/F1 công bằng
    return dict(kpi=gk.kpi.iloc[0][:8], n_anom_test=int(y[tm].sum()),
                order='(' + str(P) + ',' + str(d) + ',' + str(Q) + ')', period=period,
                aic=round(float(aic), 1),
                AP=round(r['AP_pw'], 3), AP_val=round(r['AP_val'], 3), ROC=round(r['ROC'], 3),
                PW_P=round(r['PW']['precision'], 3), PW_R=round(r['PW']['recall'], 3),
                PW_F1=round(r['PW']['fbeta'], 3),
                thr_pw=round(r['PW']['threshold'], 4), ttd_s=r['TTD']['ttd_sec_mean'], path=path,
                _score=score[tm], _ts=p.timestamp.values[tm], _y=y[tm])


In [5]:
rows, skipped, SCORES = [], [], []
t0 = time.time()
for kpi, gk in df.groupby('kpi'):
    try:
        r = run_kpi(gk.copy())
    except Exception as e:
        r = None
        print('  lỗi', kpi[:8], type(e).__name__)
    if r is None:
        skipped.append(kpi[:8])
    else:
        SCORES.append(pd.DataFrame({'kpi': r['kpi'], 'timestamp': r.pop('_ts'),
                                    'y': r.pop('_y'), 'score': r.pop('_score')}))
        rows.append(r)
        print('done', r['kpi'], '| order', r['order'], '| AP', r['AP'],
              '| PW_F1', r['PW_F1'], '|', r['path'])
print()
print('Skipped (' + str(len(skipped)) + '):', skipped)
print('Tổng thời gian: ' + str(round(time.time() - t0)) + 's')

done 02e99bd4 | order (3,0,1) | AP 0.473 | PW_F1 0.482 | sm
done 07927a9a | order (3,0,2) | AP 0.077 | PW_F1 0.0 | sm
done 09513ae3 | order (2,1,2) | AP 0.013 | PW_F1 0.0 | sm
done 18fbb1d5 | order (3,0,1) | AP 0.485 | PW_F1 0.004 | sm
done 1c35dbf5 | order (2,0,1) | AP 0.292 | PW_F1 0.334 | sm
done 40e25005 | order (3,0,1) | AP 0.234 | PW_F1 0.29 | sm
done 71595dd7 | order (2,0,2) | AP 0.176 | PW_F1 0.258 | sm
done 7c189dd3 | order (2,0,2) | AP 0.699 | PW_F1 0.704 | sm
done 88cf3a77 | order (3,0,1) | AP 0.242 | PW_F1 0.224 | sm
done 8bef9af9 | order (2,0,2) | AP 0.554 | PW_F1 0.581 | sm
done 8c892e55 | order (3,0,1) | AP 0.194 | PW_F1 0.121 | sm
done 9ee58794 | order (2,0,2) | AP 0.855 | PW_F1 0.776 | sm
done a40b1df8 | order (2,0,2) | AP 0.722 | PW_F1 0.744 | sm
done affb01ca | order (2,0,2) | AP 0.615 | PW_F1 0.615 | sm
done c58bfcba | order (2,0,2) | AP 0.008 | PW_F1 0.0 | sm
done cff6d3c0 | order (3,0,1) | AP 0.177 | PW_F1 0.24 | sm
done da403e4e | order (2,0,2) | AP 0.752 | PW_F1

## Bảng kết quả per-KPI + macro

In [6]:
res = pd.DataFrame(rows).sort_values('AP', ascending=False).reset_index(drop=True)
n_all = df.kpi.nunique()
mAP = res['AP'].mean(); mPW = res['PW_F1'].mean()
print(f'MACRO AP    = {mAP:.3f}  (trên {len(res)}/{n_all} KPI)')
print(f'MACRO PW_F1 = {mPW:.3f} | MACRO ROC = {res["ROC"].mean():.3f} | MACRO AP_val = {res["AP_val"].mean():.3f}')
print('Phân bố order (p,d,q):', res['order'].value_counts().to_dict())
print('Phân bố path:', res['path'].value_counts().to_dict())
res

MACRO AP    = 0.381  (trên 18/26 KPI)
MACRO PW_F1 = 0.312 | MACRO ROC = 0.795 | MACRO AP_val = 0.377
Phân bố order (p,d,q): {'(2,0,2)': 9, '(3,0,1)': 6, '(2,0,1)': 1, '(3,0,2)': 1, '(2,1,2)': 1}
Phân bố path: {'sm': 18}


,kpi,n_anom_test,order,period,aic,AP,AP_val,ROC,PW_P,PW_R,PW_F1,thr_pw,ttd_s,path
0,9ee58794,761,"(2,0,2)",1440,-14727.3,0.855,0.175,0.952,0.727,0.832,0.776,6.0154,24.000000,sm
1,da403e4e,238,"(2,0,2)",1440,6715.0,0.752,0.691,0.930,0.065,0.857,0.121,1.6636,7.500000,sm
2,a40b1df8,86,"(2,0,2)",1440,-9025.6,0.722,0.690,0.920,0.829,0.674,0.744,6.2203,63.000000,sm
3,7c189dd3,63,"(2,0,2)",1440,-9195.2,0.699,0.702,0.919,0.844,0.603,0.704,7.5148,64.615385,sm
4,affb01ca,76,"(2,0,2)",1440,-9208.4,0.615,0.670,0.868,0.878,0.474,0.615,8.7279,84.705882,sm
5,8bef9af9,73,"(2,0,2)",1440,-8800.9,0.554,0.677,0.835,0.773,0.466,0.581,7.6858,85.714286,sm
6,18fbb1d5,47,"(3,0,1)",1440,7604.9,0.485,0.193,0.876,0.002,1.000,0.004,0.0008,0.000000,sm
7,02e99bd4,1382,"(3,0,1)",1440,1725.0,0.473,0.286,0.789,0.627,0.392,0.482,3.7806,100.000000,sm
8,e0770391,2706,"(2,0,2)",1440,-9250.1,0.292,0.165,0.779,0.553,0.065,0.117,4.6520,481.034483,sm
9,1c35dbf5,2709,"(2,0,1)",1440,12060.4,0.292,0.692,0.691,0.314,0.355,0.334,1.9742,132.000000,sm


In [7]:
out_path = ROOT / 'Modeling' / 'Stats' / 'ARIMA' / 'arima_per_kpi_config.json'
res.to_json(out_path, orient='records', indent=1)
ART = ROOT / 'Modeling' / 'Artifacts'; ART.mkdir(parents=True, exist_ok=True)
pd.concat(SCORES, ignore_index=True).to_parquet(ART / 'scores_ARIMA.parquet', index=False)
print('Đã lưu config per-KPI:', out_path, '| score ->', ART / 'scores_ARIMA.parquet')

Đã lưu config per-KPI: C:\Projects\anomaly-detection-fundamentals\Modeling\Stats\ARIMA\arima_per_kpi_config.json | score -> C:\Projects\anomaly-detection-fundamentals\Modeling\Artifacts\scores_ARIMA.parquet
